# 🛰️ Remote Sensing Change Detection — Google Colab T4 Eğitim Defteri

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/osmankalkan/remote-sensing-change-detection/blob/main/notebooks/colab_train.ipynb)

> **Hedef:** Google Colab ortamında (NVIDIA T4 GPU) depoyu klonlamak/güncellemek, Google Drive'ı bağlamak, gerekli kütüphaneleri kurmak, `src/train.py` eğitim scriptini arka planda çalıştırmak ve en iyi model ağırlıklarını (`best_model.pth`) doğrudan Google Drive'a kalıcı olarak kaydetmek.

### 📋 İş Akışı Adımları:
1. **GPU & Ortam Kontrolü:** NVIDIA T4 GPU ve CUDA doğrulaması.
2. **Google Drive Bağlantısı:** Ağırlıkların ve logların kaydedileceği kalıcı Drive dizini oluşturma.
3. **GitHub Depo Yönetimi:** Repoyu klonlama veya en son değişiklikleri `git pull` ile çekme.
4. **Bağımlılıkların Kurulumu:** `requirements.txt` ve Kaggle CLI kütüphanelerinin yüklenmesi.
5. **Veri Seti Hazırlığı (Kaggle API):** `kaggle.json` ile kimlik doğrulama, `adrianblanchon/levir-cd-256` veri setini indirme ve `data/processed/{train,val,test}/{A,B,label}` hiyerarşisine yerleştirme.
6. **Eğitim Konfigürasyonu:** T4 GPU için optimize edilmiş `config_colab.yaml` oluşturma.
7. **Arka Planda Eğitim (`nohup`):** `src/train.py` sürecini arka planda başlatma.
8. **Canlı İzleme & Metrik Takibi:** Canlı logları ve IoU/F1 değerlerini izleme (Hücre kesilse dahi eğitim devam eder).
9. **Google Drive Doğrulaması:** `best_model.pth` kontrolü ve doğrulama görselleştirmelerini inceleme.
10. **Yardımcı Araçlar:** Süreç yönetimi ve GPU bellek temizleme.

## 1. GPU ve Donanım Kontrolü
T4 GPU'nun bağlı olduğunu ve CUDA'nın aktif olduğunu doğrulayalım.
*(Colab Menü: **Çalışma Zamanı** -> **Çalışma zamanı türünü değiştir** -> **T4 GPU**)*

In [ ]:
import os
import torch
import subprocess

print("=" * 60)
print("🚀 DONANIM VE SİSTEM KONTROLÜ")
print("=" * 60)

cuda_ok = torch.cuda.is_available()
print(f"CUDA Kullanılabilir mi?: {cuda_ok}")

if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Aktif GPU: {gpu_name}")
    print(f"Toplam VRAM: {vram_gb:.2f} GB")
    if "T4" in gpu_name:
        print("✅ NVIDIA Tesla T4 GPU aktif! (T4 16GB VRAM change detection için idealdir)")
    else:
        print(f"ℹ️ {gpu_name} GPU aktif.")
else:
    print("⚠️ UYARI: GPU algılanamadı!")
    print("Lütfen Colab menüsünden: Runtime -> Change runtime type -> T4 GPU seçin.")

!nvidia-smi

## 2. Google Drive Bağlantısı (Mount)
Model ağırlıklarının (`best_model.pth`), eğitim loglarının ve doğrulama görsellerinin Colab oturumu kapansa dahi kaybolmaması için Google Drive'ı bağlıyoruz.

In [ ]:
from google.colab import drive
import os

print("📂 Google Drive bağlanıyor...")
drive.mount('/content/drive')

# Google Drive üzerinde projeye özel kalıcı klasör
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/remote_sensing_change_detection"
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, "checkpoints")
DRIVE_VIS_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, "visualizations")

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_VIS_DIR, exist_ok=True)

print("\n✅ Google Drive başarıyla bağlandı!")
print(f"📁 Model Ağırlıkları Kayıt Dizini: {DRIVE_CHECKPOINT_DIR}")
print(f"🖼️ Doğrulama Görselleri Dizini: {DRIVE_VIS_DIR}")

## 3. GitHub Deposunu Klonlama veya Güncelleme
Proje dizini mevcut değilse `git clone` yapılır, mevcutsa `git pull` ile en güncel kodlar çekilir.

In [ ]:
import os

REPO_URL = "https://github.com/osmankalkan/remote-sensing-change-detection.git"
LOCAL_REPO_DIR = "/content/remote-sensing-change-detection"
BRANCH = "main"

if os.path.exists(LOCAL_REPO_DIR):
    print(f"🔄 Depo zaten mevcut: {LOCAL_REPO_DIR}")
    print(f"📥 Güncellemeler alınıyor (git fetch & pull)...")
    %cd {LOCAL_REPO_DIR}
    !git fetch origin
    !git reset --hard origin/{BRANCH} || !git pull origin {BRANCH}
else:
    print(f"📥 Depo klonlanıyor: {REPO_URL}...")
    %cd /content
    !git clone {REPO_URL}
    %cd {LOCAL_REPO_DIR}

print("\n✅ Aktif Çalışma Dizini:")
%pwd

## 4. Gerekli Kütüphanelerin Kurulumu
`requirements.txt` dosyasındaki kütüphaneleri (PyTorch, Albumentations, segmentation-models-pytorch vb.) kuruyoruz.

In [ ]:
%cd {LOCAL_REPO_DIR}
print("📦 Kütüphaneler kuruluyor (requirements.txt)...\n")
!pip install -q -r requirements.txt
!pip install -q kaggle gdown huggingface_hub

# Kurulum Doğrulama
import segmentation_models_pytorch as smp
import albumentations as A
import cv2
import yaml

print("=" * 60)
print("✅ Kütüphane kurulumları başarıyla tamamlandı:")
print(f" - PyTorch: {torch.__version__} (CUDA: {torch.version.cuda})")
print(f" - Segmentation Models PyTorch: {smp.__version__}")
print(f" - Albumentations: {A.__version__}")
print(f" - OpenCV: {cv2.__version__}")
print("=" * 60)

## 5. Veri Seti Hazırlığı & Kontrolü (Kaggle API: LEVIR-CD-256)
Bu adımda 256x256 boyutlu yamalanmış **LEVIR-CD-256** veri seti doğrudan Kaggle API üzerinden indirilir:

1. **/root/.kaggle/kaggle.json** kontrol edilir; yoksa `files.upload()` ile yüklemeniz istenir ve `chmod 600` yetkisi verilir.
2. `kaggle datasets download -d adrianblanchon/levir-cd-256 -p /content/data_temp --unzip` komutuyla veri seti doğrudan indirilir ve açılır.
3. İnen veriler projenin beklediği `data/processed/{train,val,test}/{A,B,label}` hiyerarşisine uygun olacak şekilde `data/processed` altına taşınır ve geçici klasör temizlenir.
4. Dosyaların başarıyla yerleştiğini doğrulamak için `train/A`, `train/B` ve `train/label` içindeki dosya sayıları ekrana yazdırılır.

In [ ]:
import os
import shutil
import glob
from pathlib import Path
from google.colab import files

print("=" * 65)
print("📥 5. VERİ SETİ HAZIRLIĞI (KAGGLE API: LEVIR-CD-256)")
print("=" * 65)

# -------------------------------------------------------------------
# 1. Kaggle API Kimlik Doğrulaması (kaggle.json)
# -------------------------------------------------------------------
kaggle_dir = "/root/.kaggle"
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(kaggle_json_path):
    print("🔑 /root/.kaggle/kaggle.json bulunamadı.")
    print("👉 Lütfen Kaggle hesabınızdan indirdiğiniz 'kaggle.json' dosyasını yükleyin:")
    uploaded = files.upload()
    
    found_key = False
    for filename in uploaded.keys():
        if filename == "kaggle.json" or filename.endswith(".json"):
            src_upload = os.path.join("/content", filename) if os.path.exists(os.path.join("/content", filename)) else filename
            shutil.move(src_upload, kaggle_json_path)
            found_key = True
            break
            
    if not found_key and not os.path.exists(kaggle_json_path):
        raise FileNotFoundError("❌ kaggle.json dosyası yüklenemedi! Lütfen geçerli bir Kaggle API anahtarı yükleyin.")

# Gerekli dosya izinlerini ayarla (chmod 600)
os.chmod(kaggle_json_path, 0o600)
print("✅ kaggle.json başarıyla doğrulandı ve izinleri (chmod 600) ayarlandı.\n")

# -------------------------------------------------------------------
# 2. Kaggle'dan Veri Setini İndir ve Aç
# -------------------------------------------------------------------
temp_download_dir = "/content/data_temp"
processed_dir = "data/processed"
train_a_path = os.path.join(processed_dir, "train", "A")

already_downloaded = os.path.exists(train_a_path) and len(os.listdir(train_a_path)) > 0

if already_downloaded:
    print(f"ℹ️ Veri seti zaten mevcut ({train_a_path} içerisinde {len(os.listdir(train_a_path))} dosya var).")
    print("Yeniden indirmek isterseniz data/processed klasörünü silip bu hücreyi tekrar çalıştırabilirsiniz.\n")
else:
    if os.path.exists(temp_download_dir):
        shutil.rmtree(temp_download_dir)
    os.makedirs(temp_download_dir, exist_ok=True)
    
    print("🚀 Kaggle'dan 'adrianblanchon/levir-cd-256' indiriliyor ve açılıyor...")
    !kaggle datasets download -d adrianblanchon/levir-cd-256 -p /content/data_temp --unzip
    print("✅ İndirme ve arşivden çıkarma tamamlandı.\n")

    # -------------------------------------------------------------------
    # 3. İnen Verileri data/processed/{train,val,test}/{A,B,label} Yapısına Taşı
    # -------------------------------------------------------------------
    print("🚚 Dosyalar data/processed/{train,val,test}/{A,B,label} yapısına organize ediliyor...")
    os.makedirs(processed_dir, exist_ok=True)

    # Kaynak kök dizinini tespit et
    source_root = temp_download_dir
    for root, dirs, _ in os.walk(temp_download_dir):
        if any(d in dirs for d in ["train", "val", "test"]) or ("A" in dirs and "B" in dirs):
            source_root = root
            break

    print(f"📂 Kaynak veri dizini tespit edildi: {source_root}")

    def move_subfolders(src_dir, split_name):
        target_split_dir = os.path.join(processed_dir, split_name)
        os.makedirs(target_split_dir, exist_ok=True)
        
        for sub in ["A", "B", "label"]:
            target_sub = os.path.join(target_split_dir, sub)
            os.makedirs(target_sub, exist_ok=True)
            
            # A, B, label, labels, Label gibi varyasyonları kontrol et
            candidates = [sub, sub.lower(), sub.upper(), sub + "s", sub.capitalize()]
            src_sub = None
            for c in candidates:
                cand_path = os.path.join(src_dir, c)
                if os.path.isdir(cand_path):
                    src_sub = cand_path
                    break
            
            if src_sub:
                for fname in os.listdir(src_sub):
                    sf = os.path.join(src_sub, fname)
                    tf = os.path.join(target_sub, fname)
                    if os.path.isfile(sf):
                        shutil.move(sf, tf)

    found_splits = [d for d in ["train", "val", "test"] if os.path.isdir(os.path.join(source_root, d))]
    if found_splits:
        for split in found_splits:
            move_subfolders(os.path.join(source_root, split), split)
    else:
        # Eğer split klasörleri yoksa, list/ altındaki dosya listelerine bak
        txt_files = glob.glob(os.path.join(source_root, "**", "*.txt"), recursive=True)
        split_lists = {}
        for tf in txt_files:
            s_name = Path(tf).stem.lower()
            if s_name in ["train", "val", "test"]:
                with open(tf, "r") as f:
                    split_lists[s_name] = set(line.strip() for line in f if line.strip())

        src_subs = {}
        for sub in ["A", "B", "label"]:
            for c in [sub, sub.lower(), sub.upper(), sub + "s"]:
                matches = [m for m in glob.glob(os.path.join(source_root, "**", c), recursive=True) if os.path.isdir(m)]
                if matches:
                    src_subs[sub] = matches[0]
                    break

        if split_lists and "A" in src_subs and "B" in src_subs:
            for split, fnames in split_lists.items():
                for sub, s_folder in src_subs.items():
                    t_folder = os.path.join(processed_dir, split, sub)
                    os.makedirs(t_folder, exist_ok=True)
                    for fn in fnames:
                        base = fn if os.path.splitext(fn)[1] else fn + ".png"
                        src_f = os.path.join(s_folder, base)
                        if not os.path.exists(src_f):
                            cand = glob.glob(os.path.join(s_folder, f"{Path(fn).stem}.*"))
                            if cand:
                                src_f = cand[0]
                        if os.path.exists(src_f):
                            shutil.move(src_f, os.path.join(t_folder, os.path.basename(src_f)))
        else:
            # Klasörler tek havuzdaysa 80/10/10 olarak böl
            if "A" in src_subs and "B" in src_subs:
                all_names = sorted(os.listdir(src_subs["A"]))
                n = len(all_names)
                nt, nv = int(n * 0.8), int(n * 0.1)
                splits = {"train": all_names[:nt], "val": all_names[nt:nt+nv], "test": all_names[nt+nv:]}
                for split, fnames in splits.items():
                    for sub, s_path in src_subs.items():
                        t_folder = os.path.join(processed_dir, split, sub)
                        os.makedirs(t_folder, exist_ok=True)
                        for fn in fnames:
                            sf = os.path.join(s_path, fn)
                            if os.path.exists(sf):
                                shutil.move(sf, os.path.join(t_folder, fn))

    # Geçici klasörü temizle
    if os.path.exists(temp_download_dir):
        shutil.rmtree(temp_download_dir, ignore_errors=True)
        print("🧹 Geçici indirme klasörü (/content/data_temp) temizlendi.\n")

# -------------------------------------------------------------------
# 4. Dosyaların Başarıyla Yerleştiğini Doğrula
# -------------------------------------------------------------------
train_a_count = len(os.listdir("data/processed/train/A")) if os.path.exists("data/processed/train/A") else 0
train_b_count = len(os.listdir("data/processed/train/B")) if os.path.exists("data/processed/train/B") else 0
train_lbl_count = len(os.listdir("data/processed/train/label")) if os.path.exists("data/processed/train/label") else 0

val_a_count = len(os.listdir("data/processed/val/A")) if os.path.exists("data/processed/val/A") else 0
val_b_count = len(os.listdir("data/processed/val/B")) if os.path.exists("data/processed/val/B") else 0
val_lbl_count = len(os.listdir("data/processed/val/label")) if os.path.exists("data/processed/val/label") else 0

test_a_count = len(os.listdir("data/processed/test/A")) if os.path.exists("data/processed/test/A") else 0
test_b_count = len(os.listdir("data/processed/test/B")) if os.path.exists("data/processed/test/B") else 0
test_lbl_count = len(os.listdir("data/processed/test/label")) if os.path.exists("data/processed/test/label") else 0

print("=" * 65)
print("📊 VERİ SETİ DOĞRULAMA RAPORU (data/processed)")
print("=" * 65)
print(f"📁 [TRAIN] -> A: {train_a_count} adet | B: {train_b_count} adet | label: {train_lbl_count} adet")
print(f"📁 [VAL]   -> A: {val_a_count} adet | B: {val_b_count} adet | label: {val_lbl_count} adet")
if test_a_count > 0:
    print(f"📁 [TEST]  -> A: {test_a_count} adet | B: {test_b_count} adet | label: {test_lbl_count} adet")
print("=" * 65)

if train_a_count > 0 and train_a_count == train_b_count == train_lbl_count:
    print("✅ MÜKEMMEL: train/A, train/B ve train/label dosya sayıları tam eşleşiyor!")
    print("🚀 Veri seti eğitime hazır.")
elif train_a_count > 0:
    print("⚠️ Dikkat: train klasöründe A, B ve label dosya sayıları arasında fark var, kontrol ediniz.")
else:
    print("❌ Hata: data/processed/train/A klasöründe dosya bulunamadı!")

## 6. Colab T4 GPU Eğitim Konfigürasyonu
Google Drive kayıt dizini (`save_dir`), batch size, epoch sayısı ve CUDA cihazı Colab T4 için ayarlanır.

In [ ]:
import yaml
import os

with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Colab T4 GPU Ayarları
config["training"]["device"] = "cuda"
config["training"]["batch_size"] = 8          # T4 GPU için 8 veya 16 önerilir
config["training"]["epochs"] = 30
config["training"]["save_dir"] = DRIVE_CHECKPOINT_DIR
config["data"]["num_workers"] = 2

colab_config_path = "configs/config_colab.yaml"
with open(colab_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Colab konfigürasyonu kaydedildi: {colab_config_path}")
print(f" - Model Tipi: {config['model']['type']} ({config['model']['encoder_name']})")
print(f" - Batch Size: {config['training']['batch_size']}")
print(f" - Epoch Sayısı: {config['training']['epochs']}")
print(f" - Kayıt Dizini (Google Drive): {config['training']['save_dir']}")

## 7. `src/train.py` Eğitim Scriptini Arka Planda Başlatma
Eğitim süreci `nohup` ve `&` ile arka planda çalıştırılır.
Bu sayede:
- Tarayıcı sekmesi kazara kapansa veya bağlantı kopsa bile eğitim Colab VM üzerinde kesintisiz devam eder.
- Çıktılar hem yerel `train.log` dosyasına yazılır hem de Colab konsolundan takip edilebilir.
- En iyi model ağırlıkları (`best_model.pth`) her IoU rekorunda doğrudan Google Drive'a kaydedilir.

In [ ]:
import subprocess
import time
import os

log_file_path = "/content/remote-sensing-change-detection/train.log"

# Varsa önceki eğitim logunu temizle
if os.path.exists(log_file_path):
    os.remove(log_file_path)

# Arka plan çalıştırma komutu (unbuffered Python ile anlık log akışı)
cmd = (
    f"nohup python -u src/train.py "
    f"--config configs/config_colab.yaml "
    f"--save_dir '{DRIVE_CHECKPOINT_DIR}' "
    f"> '{log_file_path}' 2>&1 &"
)

print("🚀 Eğitim arka planda başlatılıyor...")
os.system(cmd)
time.sleep(3)

# Süreç kontrolü
ps_output = subprocess.getoutput("ps aux | grep 'src/train.py' | grep -v 'grep'")
if ps_output:
    print("=" * 65)
    print("✅ EĞİTİM ARKA PLANDA BAŞARIYLA ÇALIŞIYOR!")
    print("=" * 65)
    print(f"Süreç Bilgisi:\n{ps_output.strip()}")
    print(f"\n📝 Canlı Log Dosyası: {log_file_path}")
    print(f"💾 Hedef Checkpoint: {DRIVE_CHECKPOINT_DIR}/best_model.pth")
else:
    print("⚠️ Süreç başlatılamadı. Hata detayları:")
    if os.path.exists(log_file_path):
        with open(log_file_path, "r") as f:
            print(f.read())

## 8. Canlı Eğitim Takibi & Metrik Monitörü
Aşağıdaki hücre arka plandaki `train.log` dosyasını canlı olarak okur ve ekrana basar.
> 💡 **Önemli:** Bu hücreyi durdurmak (⏹️ Interrupt Kernel) arka plandaki eğitimi **durdurmaz**. İstediğiniz zaman hücreyi durdurup tekrar çalıştırabilirsiniz.

In [ ]:
import time
import os
import subprocess

print("👀 Canlı Log Akışı Başlatıldı (Durdurmak için hücreyi 'Interrupt' edin)\n" + "-" * 60)

log_file_path = "/content/remote-sensing-change-detection/train.log"

# Log dosyasının oluşmasını bekle
for _ in range(10):
    if os.path.exists(log_file_path) and os.path.getsize(log_file_path) > 0:
        break
    time.sleep(1)

try:
    with open(log_file_path, "r") as f:
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                # Süreç hala çalışıyor mu?
                is_running = subprocess.getoutput("pgrep -f 'src/train.py'")
                if not is_running:
                    print("\n" + "=" * 60)
                    print("🏁 Eğitim süreci tamamlandı!")
                    print("=" * 60)
                    break
                time.sleep(1)
except KeyboardInterrupt:
    print("\n\n⏸️ Canlı log izleme kullanıcı tarafından durduruldu.")
    print("ℹ️ Eğitim arka planda kesintisiz devam ediyor.")

## 9. Google Drive Doğrulaması & Kaydedilen Model Ağırlıkları
Google Drive'daki `best_model.pth` dosyasını inceler, ulaşılan en iyi IoU / F1 skorlarını ve görselleştirme çıktılarını gösterir.

In [ ]:
import os
import torch
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

best_model_path = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pth")
latest_model_path = os.path.join(DRIVE_CHECKPOINT_DIR, "latest_model.pth")

print("=" * 65)
print("🔍 GOOGLE DRIVE CHECKPOINT RAPORU")
print("=" * 65)

if os.path.exists(best_model_path):
    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
    print(f"✅ En İyi Model: {best_model_path} ({size_mb:.2f} MB)")
    
    ckpt = torch.load(best_model_path, map_location="cpu")
    print(f"📌 En İyi Epoch: {ckpt.get('epoch', 'Bilinmiyor')}")
    scores = ckpt.get("val_scores", {})
    print(f"🎯 Val IoU:        {scores.get('iou', 0):.4f}")
    print(f"🎯 Val F1-Skor:    {scores.get('f1', 0):.4f}")
    print(f"🎯 Val Hassasiyet: {scores.get('precision', 0):.4f}")
    print(f"🎯 Val Duyarlılık: {scores.get('recall', 0):.4f}")
else:
    print("⏳ best_model.pth henüz kaydedilmedi (Eğitim henüz ilk epochlarda olabilir).")

# Varsa görselleştirme snapshot'larını göster
vis_dir = os.path.join(DRIVE_CHECKPOINT_DIR, "visualizations")
vis_images = sorted(glob.glob(os.path.join(vis_dir, "*.png")))

if vis_images:
    latest_img_path = vis_images[-1]
    print(f"\n🖼️ Son Doğrulama Görselleştirmesi: {latest_img_path}")
    img = mpimg.imread(latest_img_path)
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Doğrulama Tahmini - {os.path.basename(latest_img_path)}", fontsize=12, weight='bold')
    plt.show()
else:
    print(f"ℹ️ Henüz görselleştirme üretilmedi (Her 5 epochta bir kaydedilir).")

## 10. Yardımcı Araçlar (Süreç Durdurma & GPU Belleği Temizleme)
Gerektiğinde arka plandaki eğitimi durdurmak veya GPU önbelleğini temizlemek için bu hücreyi kullanabilirsiniz.

In [ ]:
# 🛑 Arka plandaki eğitimi durdurmak için aşağıdaki satırın yorumunu kaldırıp çalıştırın:
# !pkill -f 'src/train.py'
# print("Eğitim süreci durduruldu.")

# 🧹 GPU Önbelleğini Temizlemek İsterseniz:
# import torch
# torch.cuda.empty_cache()
# print("GPU önbelleği boşaltıldı.")